# GIẢI THUẬT 3: BAYES NGÂY THƠ (NAÏVE BAYES)

## 2.3.1. Ôn tập lý thuyết


### 1. Cách hoạt động:

$$
P(y\mid x_1,\ldots,x_n)=\frac{P(y)\,P(x_1,\ldots,x_n\mid y)}{P(x_1,\ldots,x_n)}
$$

- $P(y)$:  xác suất tiên nghiệm (prior) của lớp y, tính từ tần suất lớp trong tập huấn luyện.
- $P(x\mid y)$: likelihood, xác suất quan sát được đặc trưng x khi biết lớp y.
- $P(y\mid x)$: xác suất hậu nghiệm (posterior), là thứ ta cần để dự đoán.
- $P(x)$: mẫu số, giống nhau với mọi lớp nên có thể bỏ qua khi so sánh.

Giả định ngây thơ: các đặc trưng độc lập có điều kiện với nhau khi biết lớp:

$$
P(x_1,\ldots,x_n\mid y)=\prod_{i=1}^{n}P(x_i\mid y)
$$

Nhờ đó, thay vì ước lượng một phân phối đồng thời nhiều chiều (cần lượng dữ liệu khổng lồ), ta chỉ cần ước lượng n phân phối một chiều. Quy tắc dự đoán:

$$
\hat{y}=\operatorname*{arg\,max}_{y}\left[\log P(y)+\sum_i\log P(x_i\mid y)\right]
$$

Ta dùng log để tránh underflow khi nhân nhiều xác suất nhỏ. Ngoài ra cần Laplace smoothing (tham số alpha) để tránh trường hợp một giá trị chưa từng xuất hiện trong lớp làm xác suất bằng 0 và triệt tiêu cả tích.


### 2. Ba loại mô hình Naive Bayes

| Mô hình | Kiểu đặc trưng | Cách mô hình hóa P(xᵢ\|y) |
| --- | --- | --- |
| Gaussian NB | Liên tục (số thực) | Phân phối chuẩn, ước lượng μ và σ² theo từng lớp |
| Multinomial NB | Đếm (số nguyên không âm) | Phân phối đa thức, dựa trên tần suất xuất hiện |
| Bernoulli NB | Nhị phân (0/1) | Phân phối Bernoulli, ghi nhận cả sự vắng mặt của đặc trưng |

Điểm khác biệt quan trọng giữa Multinomial và Bernoulli: Bernoulli phạt cả khi một từ không xuất hiện, còn Multinomial chỉ quan tâm các từ có xuất hiện. Ngoài ra scikit-learn còn có CategoricalNB cho đặc trưng phân loại rời rạc, và ComplementNB phù hợp với dữ liệu mất cân bằng lớp.


### 3. Vì sao gọi là “ngây thơ”?

Vì giả định các đặc trưng độc lập với nhau (khi biết lớp) hầu như không đúng trong thực tế. Ví dụ trong văn bản, từ “máy” và “tính” thường đi cùng nhau, còn trong dữ liệu nhà đất, diện tích và số phòng tương quan mạnh.

**Ảnh hưởng đến hiệu suất:**

- Xác suất bị sai lệch: khi các đặc trưng tương quan, mô hình “đếm trùng” bằng chứng, nên xác suất đầu ra thường quá tự tin (gần 0 hoặc 1). Vì vậy không nên tin predict_proba như xác suất đã được hiệu chỉnh (calibrated).
- Nhãn dự đoán vẫn thường đúng: bài toán phân loại chỉ cần lớp có xác suất cao nhất, không cần giá trị xác suất chính xác. Vì thế Naive Bayes vẫn hoạt động tốt một cách đáng ngạc nhiên dù giả định bị vi phạm.
- Hiệu suất giảm rõ khi có nhiều đặc trưng trùng lặp hoặc tương quan rất mạnh, hoặc khi ranh giới quyết định phụ thuộc vào tương tác giữa các đặc trưng.


### 4. Ưu điểm và hạn chế so với SVM, Random Forest

**Ưu điểm:**

- Huấn luyện và dự đoán rất nhanh (chỉ cần đếm và tính thống kê), độ phức tạp tuyến tính theo dữ liệu.
- Hoạt động tốt với dữ liệu ít và dữ liệu nhiều chiều (như văn bản).
- Ít siêu tham số, dễ cài đặt, dễ giải thích, hỗ trợ học tăng dần (partial_fit).
- Ít bị overfit khi dữ liệu nhỏ.

**Hạn chế:**

- Giả định độc lập làm giảm độ chính xác khi đặc trưng tương quan.
- Không học được tương tác giữa các đặc trưng, nên thường kém SVM và Random Forest trên dữ liệu phức tạp.
- Xác suất đầu ra kém tin cậy.
- Gaussian NB giả định phân phối chuẩn, có thể không phù hợp với dữ liệu lệch hoặc đa đỉnh.
- Vấn đề tần suất bằng 0 nếu không smoothing.


### 5. Code mẫu Gaussian Naive Bayes với Scikit-learn

Các bước: nạp dữ liệu → chia train/test → khởi tạo và huấn luyện mô hình → dự đoán → đánh giá.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Nạp dữ liệu và chia train/test
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# 2. Huấn luyện
model = GaussianNB()
model.fit(X_train, y_train)

# 3. Dự đoán và đánh giá
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Xác suất hậu nghiệm cho mẫu đầu tiên
print(model.predict_proba(X_test[:1]))


Gaussian NB không cần chuẩn hóa dữ liệu vì nó ước lượng μ và σ riêng cho từng đặc trưng. Tham số var_smoothing (mặc định 1e-9) dùng để ổn định số học.


### 6. Xử lý dữ liệu phân loại trước khi dùng Multinomial NB

MultinomialNB yêu cầu đặc trưng là số không âm (thường là số đếm), nên chuỗi như “Hà Nội”, “TP.HCM” phải được mã hóa trước. Có ba cách:

- OneHotEncoder (nên dùng): mỗi giá trị thành một cột 0/1, không tạo ra thứ tự giả.
- OrdinalEncoder / LabelEncoder: chuyển thành 0, 1, 2... nhưng Multinomial NB sẽ hiểu đây là “số đếm”, nên chỉ hợp lý khi các giá trị thực sự có thứ tự.
- CategoricalNB: nếu dữ liệu toàn là đặc trưng phân loại, dùng luôn mô hình này kết hợp với OrdinalEncoder, vì nó được thiết kế riêng cho trường hợp đó.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import MultinomialNB

df = pd.DataFrame({
    "thanh_pho": ["HN", "HCM", "HN", "DN", "HCM", "DN"],
    "loai_nha": ["chung cu", "nha pho", "nha pho", "chung cu", "chung cu", "nha pho"],
    "mua": [1, 0, 1, 0, 1, 0],
})
X, y = df[["thanh_pho", "loai_nha"]], df["mua"]


In [ ]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["thanh_pho", "loai_nha"])
])
clf = Pipeline([("pre", pre), ("nb", MultinomialNB(alpha=1.0))])
clf.fit(X, y)
print(clf.predict(pd.DataFrame({"thanh_pho": ["HN"], "loai_nha": ["nha pho"]})))


Dùng Pipeline giúp bước mã hóa được fit chỉ trên tập train, tránh rò rỉ dữ liệu (data leakage).


### 7. Naive Bayes cho phân loại văn bản

**Quy trình:**

1. Tiền xử lý: chuyển chữ thường, loại bỏ ký tự đặc biệt, tách từ. Với tiếng Việt nên dùng bộ tách từ như underthesea hoặc pyvi, vì từ ghép như “học máy” cần được giữ liền.
2. Vector hóa: biến văn bản thành vector số bằng CountVectorizer (đếm từ) hoặc TfidfVectorizer (trọng số TF-IDF).
3. Huấn luyện MultinomialNB (hoặc BernoulliNB/ComplementNB).
4. Đánh giá bằng accuracy, precision, recall, F1.

Về mặt toán học: với mỗi lớp, mô hình ước lượng xác suất mỗi từ xuất hiện trong lớp đó:

$$
P(w\mid y)=\frac{\operatorname{count}(w,y)+\alpha}{\sum_{w^{\prime}}\operatorname{count}(w^{\prime},y)+\alpha\lvert V\rvert}
$$

Một văn bản mới được gán vào lớp có tổng log-xác suất của các từ trong nó (cộng log prior) lớn nhất.
